In [0]:
"""
run_dq.py — Data Quality checks for Silver layer
Creates quarantine tables automatically on first run
"""

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, lit, current_timestamp
from delta.tables import DeltaTable

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Paths
S3_DELTA_SILVER = "s3://capstone-ecomm-team8/delta/silver"
QUARANTINE_PATH = f"{S3_DELTA_SILVER}/_dq_quarantine"
LOG_PATH = f"{S3_DELTA_SILVER}/_dq_log"

print("=" * 60)
print("DQ ENGINE — Silver validation before Gold")
print("Batch     : 1  (batch_id: batch_1)")
print("=" * 60)

# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def ensure_quarantine_table():
    """Create quarantine table if it doesn't exist"""
    try:
        # Check if table exists
        if DeltaTable.isDeltaTable(spark, QUARANTINE_PATH):
            print("✓ Quarantine table exists")
            return
    except:
        pass
    
    # Create empty quarantine table
    print("Creating quarantine table...")
    schema_sql = """
        CREATE TABLE IF NOT EXISTS silver._dq_quarantine (
            table_name STRING,
            check_name STRING,
            quarantine_timestamp TIMESTAMP,
            row_data STRING
        )
        USING DELTA
        LOCATION '{}'
    """.format(QUARANTINE_PATH)
    
    spark.sql(schema_sql)
    print("✓ Quarantine table created")

def ensure_log_table():
    """Create log table if it doesn't exist"""
    try:
        if DeltaTable.isDeltaTable(spark, LOG_PATH):
            print("✓ Log table exists")
            return
    except:
        pass
    
    print("Creating log table...")
    schema_sql = """
        CREATE TABLE IF NOT EXISTS silver._dq_log (
            table_name STRING,
            check_name STRING,
            total_rows BIGINT,
            bad_rows BIGINT,
            check_timestamp TIMESTAMP,
            batch_id STRING
        )
        USING DELTA
        LOCATION '{}'
    """.format(LOG_PATH)
    
    spark.sql(schema_sql)
    print("✓ Log table created")

def log_check(table_name, check_name, total, bad, batch_id="batch_1"):
    """Log a DQ check result"""
    from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
    
    status = "PASS" if bad == 0 else "FAIL"
    symbol = "✓" if bad == 0 else "✗"
    
    print(f"  [{status}] {symbol} {check_name}: {bad:,} bad rows")
    
    # Define explicit schema for Spark Connect
    log_schema = StructType([
        StructField("table_name", StringType(), False),
        StructField("check_name", StringType(), False),
        StructField("total_rows", LongType(), False),
        StructField("bad_rows", LongType(), False),
        StructField("check_timestamp", TimestampType(), False),
        StructField("batch_id", StringType(), False)
    ])
    
    # Get current timestamp from Spark SQL
    ts = spark.sql("SELECT current_timestamp()").first()[0]
    
    log_df = spark.createDataFrame([{
        "table_name": table_name,
        "check_name": check_name,
        "total_rows": total,
        "bad_rows": bad,
        "check_timestamp": ts,
        "batch_id": batch_id
    }], schema=log_schema)
    
    log_df.write.format("delta").mode("append").save(LOG_PATH)

def quarantine_and_clean(table_name, df, bad_condition, results_list):
    """Quarantine bad rows and return clean DataFrame"""
    from pyspark.sql.functions import to_json, struct
    
    bad_df = df.filter(bad_condition)
    bad_count = bad_df.count()
    
    if bad_count > 0:
        print(f"\n  QUARANTINE — {table_name}")
        print(f"    Total : {df.count():,}")
        print(f"    Bad   : {bad_count:,}  ({bad_count/df.count()*100:.1f}%)")
        
        # Get current timestamp
        ts = spark.sql("SELECT current_timestamp()").first()[0]
        
        # Save to quarantine
        quarantine_rows = bad_df.select(
            lit(table_name).alias("table_name"),
            lit("mixed_checks").alias("check_name"),
            lit(ts).alias("quarantine_timestamp"),
            to_json(struct(*df.columns)).alias("row_data")
        )
        
        quarantine_rows.write.format("delta").mode("append").save(QUARANTINE_PATH)
    
    # Return clean data
    clean_df = df.filter(~bad_condition)
    clean_count = clean_df.count()
    
    if bad_count > 0:
        print(f"    Clean : {clean_count:,}")
    else:
        print(f"  ✓ {table_name}: all checks passed — no quarantine needed")
    
    return clean_df

# ══════════════════════════════════════════════════════════════════════════════
# SETUP
# ══════════════════════════════════════════════════════════════════════════════

ensure_quarantine_table()
ensure_log_table()

all_results = []

# ══════════════════════════════════════════════════════════════════════════════
# LOAD SILVER TABLES
# ══════════════════════════════════════════════════════════════════════════════

df_orders = spark.read.format("delta").table("silver.orders")
df_items = spark.read.format("delta").table("silver.order_items")
df_payments = spark.read.format("delta").table("silver.order_payments")
df_reviews = spark.read.format("delta").table("silver.order_reviews")

# ══════════════════════════════════════════════════════════════════════════════
# TABLE 1: orders
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 60)
print("TABLE: silver.orders")
print("─" * 60)

t = []
total = df_orders.count()

# Check: no null order_id
bad = df_orders.filter(col("order_id").isNull()).count()
log_check("orders", "orders_no_null_order_id", total, bad)
t.append(("orders_no_null_order_id", total, bad))

# Check: no null customer_id
bad = df_orders.filter(col("customer_id").isNull()).count()
log_check("orders", "orders_no_null_customer_id", total, bad)
t.append(("orders_no_null_customer_id", total, bad))

# Check: no null purchase_timestamp
bad = df_orders.filter(col("order_purchase_timestamp").isNull()).count()
log_check("orders", "orders_no_null_purchase_timestamp", total, bad)
t.append(("orders_no_null_purchase_timestamp", total, bad))

# Check: valid status
valid_statuses = ["delivered", "shipped", "processing", "canceled", "unavailable", "invoiced", "created", "approved"]
bad = df_orders.filter(~col("order_status").isin(valid_statuses)).count()
log_check("orders", "orders_valid_status", total, bad)
t.append(("orders_valid_status", total, bad))

# Check: no duplicate order_id
bad = df_orders.groupBy("order_id").count().filter(col("count") > 1).count()
log_check("orders", "orders_no_duplicate_order_id", total, bad)
t.append(("orders_no_duplicate_order_id", total, bad))

# Check: approved not before purchase
bad = df_orders.filter(
    col("order_approved_at").isNotNull() &
    (col("order_approved_at") < col("order_purchase_timestamp"))
).count()
log_check("orders", "orders_approved_not_before_purchase", total, bad)
t.append(("orders_approved_not_before_purchase", total, bad))

# Check: delivered not before purchase
bad = df_orders.filter(
    col("order_delivered_customer_date").isNotNull() &
    (col("order_delivered_customer_date") < col("order_purchase_timestamp"))
).count()
log_check("orders", "orders_delivered_not_before_purchase", total, bad)
t.append(("orders_delivered_not_before_purchase", total, bad))

all_results.extend(t)

# Quarantine bad orders
orders_bad = (
    col("order_id").isNull()
    | col("customer_id").isNull()
    | col("order_purchase_timestamp").isNull()
    | ~col("order_status").isin(valid_statuses)
)
df_orders_clean = quarantine_and_clean("orders", df_orders, orders_bad, t)

# ══════════════════════════════════════════════════════════════════════════════
# TABLE 2: order_items
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 60)
print("TABLE: silver.order_items")
print("─" * 60)

t = []
total = df_items.count()

# Checks...
bad = df_items.filter(col("order_id").isNull()).count()
log_check("order_items", "order_items_no_null_order_id", total, bad)
t.append(("order_items_no_null_order_id", total, bad))

bad = df_items.filter(col("product_id").isNull()).count()
log_check("order_items", "order_items_no_null_product_id", total, bad)
t.append(("order_items_no_null_product_id", total, bad))

bad = df_items.filter(col("seller_id").isNull()).count()
log_check("order_items", "order_items_no_null_seller_id", total, bad)
t.append(("order_items_no_null_seller_id", total, bad))

bad = df_items.filter((col("price").isNull()) | (col("price") <= 0)).count()
log_check("order_items", "order_items_price_positive", total, bad)
t.append(("order_items_price_positive", total, bad))

bad = df_items.filter((col("freight_value").isNull()) | (col("freight_value") < 0)).count()
log_check("order_items", "order_items_freight_non_negative", total, bad)
t.append(("order_items_freight_non_negative", total, bad))

bad = df_items.filter(col("price") > 50000).count()
log_check("order_items", "order_items_price_not_extreme", total, bad)
t.append(("order_items_price_not_extreme", total, bad))

bad = df_items.filter((col("order_item_id").isNull()) | (col("order_item_id") <= 0)).count()
log_check("order_items", "order_items_item_id_positive", total, bad)
t.append(("order_items_item_id_positive", total, bad))

bad = df_items.groupBy("order_id", "order_item_id").count().filter(col("count") > 1).count()
log_check("order_items", "order_items_no_duplicate_line", total, bad)
t.append(("order_items_no_duplicate_line", total, bad))

all_results.extend(t)

items_bad = (
    col("order_id").isNull()
    | col("product_id").isNull()
    | col("seller_id").isNull()
    | (col("price") <= 0)
    | (col("freight_value") < 0)
)
df_items_clean = quarantine_and_clean("order_items", df_items, items_bad, t)

# ══════════════════════════════════════════════════════════════════════════════
# TABLE 3: order_payments
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 60)
print("TABLE: silver.order_payments")
print("─" * 60)

t = []
total = df_payments.count()

bad = df_payments.filter(col("order_id").isNull()).count()
log_check("order_payments", "order_payments_no_null_order_id", total, bad)
t.append(("order_payments_no_null_order_id", total, bad))

valid_types = ["credit_card", "boleto", "voucher", "debit_card", "not_defined"]
bad = df_payments.filter(~col("payment_type").isin(valid_types)).count()
log_check("order_payments", "order_payments_valid_payment_type", total, bad)
t.append(("order_payments_valid_payment_type", total, bad))

bad = df_payments.filter((col("payment_value").isNull()) | (col("payment_value") < 0)).count()
log_check("order_payments", "order_payments_value_non_negative", total, bad)
t.append(("order_payments_value_non_negative", total, bad))

bad = df_payments.filter((col("payment_installments").isNull()) | (col("payment_installments") < 1)).count()
log_check("order_payments", "order_payments_installments_at_least_one", total, bad)
t.append(("order_payments_installments_at_least_one", total, bad))

bad = df_payments.filter((col("payment_sequential").isNull()) | (col("payment_sequential") < 1)).count()
log_check("order_payments", "order_payments_sequential_at_least_one", total, bad)
t.append(("order_payments_sequential_at_least_one", total, bad))

all_results.extend(t)

payments_bad = (
    col("order_id").isNull()
    | ~col("payment_type").isin(valid_types)
    | (col("payment_value") < 0)
)
df_payments_clean = quarantine_and_clean("order_payments", df_payments, payments_bad, t)

# ══════════════════════════════════════════════════════════════════════════════
# TABLE 4: order_reviews
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 60)
print("TABLE: silver.order_reviews")
print("─" * 60)

t = []
total = df_reviews.count()

bad = df_reviews.filter(col("review_id").isNull()).count()
log_check("order_reviews", "order_reviews_no_null_review_id", total, bad)
t.append(("order_reviews_no_null_review_id", total, bad))

bad = df_reviews.filter(col("order_id").isNull()).count()
log_check("order_reviews", "order_reviews_no_null_order_id", total, bad)
t.append(("order_reviews_no_null_order_id", total, bad))

bad = df_reviews.filter(~col("review_score").between(1, 5)).count()
log_check("order_reviews", "order_reviews_valid_score", total, bad)
t.append(("order_reviews_valid_score", total, bad))

bad = df_reviews.groupBy("review_id").count().filter(col("count") > 1).count()
log_check("order_reviews", "order_reviews_no_duplicate_review_id", total, bad)
t.append(("order_reviews_no_duplicate_review_id", total, bad))

all_results.extend(t)

reviews_bad = (
    col("review_id").isNull()
    | col("order_id").isNull()
    | ~col("review_score").between(1, 5)
)
df_reviews_clean = quarantine_and_clean("order_reviews", df_reviews, reviews_bad, t)

print("\n" + "=" * 60)
print("✓ DQ checks complete")
print("=" * 60)